Adding all the tables from Dynamics 365 Business Central

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, regexp_extract

df_raw = spark.read.option("multiline", "true").format("json").load("/Volumes/workspace/erp_lakehouse/raw/cronus_usa_inc/*/customers.json").select("*", col("_metadata.file_path").alias("source_file")).withColumn("ingested_at", current_timestamp())


# Write to Bronze Table for customers the overwrite is bc the table was created using the UI. 
df_raw.write.saveAsTable("workspace.erp_lakehouse.bronze_bc_customers", mode = "overwrite", overwriteSchema = True)


table_name = {
  "items": "bronze_bc_items",
  "salesInvoices": "bronze_bc_sales_invoices",
  "salesOrders":   "bronze_bc_sales_orders"
   }


for key, value in table_name.items():
  df_raw = spark.read.option("multiline", "true").format("json").load(f"/Volumes/workspace/erp_lakehouse/raw/cronus_usa_inc/*/{key}.json").select("*", col("_metadata.file_path").alias("source_file")).withColumn("ingested_at", current_timestamp())
  df_raw.write.saveAsTable(f"workspace.erp_lakehouse.{value}", mode = "overwrite", overwriteSchema = True)




In [0]:
display(spark.table('workspace.erp_lakehouse.bronze_bc_customers'))
display(spark.table('workspace.erp_lakehouse.bronze_bc_items'))

Below is for Company B's bronze ingest.

In [0]:
company_b_tables = {
    "customers.csv": "bronze_b_customers",
    "items.csv": "bronze_b_items",
    "*/orders.csv": "bronze_b_orders",
    "*/order_lines.csv": "bronze_b_order_lines",
}

for key, value in company_b_tables.items():
    df_raw = (spark.read
        .format("csv").option("header", "true").option("inferSchema", "false").load(f"/Volumes/workspace/erp_lakehouse/raw/company_b/{key}")
        .select("*", col("_metadata.file_path").alias("source_file"))
        .withColumn("file_date", regexp_extract(col("source_file"), r"(\d{4}-\d{2}-\d{2})", 1)) #Pull the YYYY-MM-DD folder date out of the file path (blank for customers and items)
        .withColumn("ingested_at", current_timestamp()))
    df_raw.write.saveAsTable(f"workspace.erp_lakehouse.{value}", mode="overwrite", overwriteSchema=True)

In [0]:
%sql
SELECT * FROM erp_lakehouse.bronze_b_orders bol
ORDER BY bol.file_date DESC
